# The Calibrated Detector Model (v0.6)

G4LumaCam v0.6 ships with the `gaussian_probabilistic` detector model as the default,
with every parameter set to the optimum calibrated **event-by-event** against a two-second
PTB fast-neutron open-beam measurement (dataset *air45*). The calibration matched four
per-event observables (pixels per photon cluster, photons per event, pixel-event position
residual, photon time within the event) for two independent reconstructions of the same
data simultaneously, reaching $\Sigma\chi^2 = 0.52 \pm 0.02$.

| parameter | value | meaning |
|---|---|---|
| `blob` | 0.405 px | intensifier point-spread (Gaussian $\sigma$) |
| `decay_time` | 16.5 ns | P47 phosphor decay |
| `n_secondaries` | 9 | detected pixels per photon gain spot |
| `photon_keep_fraction` | 0.241 | effective optical yield ($\approx$ photocathode QE) |
| `ap_prob` | 0.012 | afterpulse satellites per photon ($\approx$2% of events) |
| `ap_rmax` / `ap_secondaries` | 5.5 px / 8 | satellite geometry (Mahon et al. 2024) |

This notebook shows how to use the calibrated defaults, how the afterpulse component
works, and how to reconstruct with the matching `empindex` presets.

## 1. Simulate and trace with the calibrated defaults

No detector arguments are needed any more - `trace_rays()` uses the calibrated model.
`calibrate=True` additionally tags every photon with its ideal (aberration-free) landing
position `x_opt`/`y_opt`, the *optical truth* used for resolution studies.

In [ ]:
import lumacam

# 1. Geant4 stage: PTB-like white fast-neutron spectrum
sim = lumacam.Simulate("v06_demo")
config = lumacam.Config.neutrons_uniform_energy()
df = sim.run(config)
df.head()

In [ ]:
# 2. Optics + detector stage: calibrated gaussian_probabilistic defaults
lens = lumacam.Lens(archive="archive/v06_demo")
lens.trace_rays(calibrate=True)   # writes tpx3Files/ + TracedPhotons/ with x_opt/y_opt

### Overriding the defaults

Every parameter can still be overridden per call. Two common cases:

In [ ]:
# a) disable the afterpulse satellites (ideal intensifier)
# lens.trace_rays(ap_prob=0)

# b) explore a different optical yield
# lens.trace_rays(photon_keep_fraction=0.35)

## 2. The afterpulse satellite component

In an MCP image intensifier a photoelectron can backscatter off the MCP input surface,
be re-accelerated across the photocathode-MCP proximity gap, and re-enter the MCP
displaced by up to $\approx 2\times$ the gap width, seeding a second, smaller avalanche.
On the camera this appears as a faint *satellite* cluster a few pixels away from the
parent cluster, in a random direction. With the conventional centroid event position the
satellite pulls the reconstructed position off the parent; anchoring the event to its
**largest cluster** removes the effect (out-of-focus position error 1.76 px $\to$ 0.66 px
$\approx$ 0.82 mm $\to$ 0.31 mm in the PTB geometry).

The satellite parameters follow R. Mahon, D. Orlov, R. Glazenborg and A. Nomerotski,
*Study of afterpulsing in optical image intensifiers*, NIM-A 1059 (2024) 168816.

## 3. Reconstruction with the calibrated `empindex` presets

The open-source [`empindex`](https://github.com/TsvikiHirsh/empindex) pipeline bundles the
calibrated reconstruction + detector model as named presets:

| preset | selection | event position |
|---|---|---|
| `best-inf` | in-focus (single-photon, $n_{px}\geq 8$) | photon position |
| `best-cog` | out-of-focus (multi-photon) | centroid |
| `best-first` | out-of-focus | earliest photon |
| `best` / `best-largest` | out-of-focus | **largest cluster (recommended)** |

On a simulation archive the preset also re-traces with the calibrated detector model
(the `trace` section); on experimental data the trace section is skipped automatically.

In [ ]:
# Reconstruct the traced archive and write a TOF-resolved TIFF stack:
#   EventImages/stack.tif  (T x 512 x 512, one page per 1.5625 ns TOF bin)
#   EventImages/sum.tif    (TOF-integrated image)
#   EventImages/tof_bins.csv
!empindex run archive/v06_demo --params best --events2image --suffix best

In [ ]:
# The in-focus reconstruction of the same trace:
!empindex run archive/v06_demo --params best-inf --events2image --suffix best_inf --tpx3-dir archive/v06_demo/trace/tpx3Files

## 4. Looking at the result

In [ ]:
import tifffile
import matplotlib.pyplot as plt
import numpy as np

img = tifffile.imread("archive/v06_demo/best/EventImages/sum.tif")
plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray_r", origin="lower")
plt.xlabel("x (image px)")
plt.ylabel("y (image px)")
plt.title("TOF-integrated event image (largest-cluster positions)")
plt.colorbar(label="events / pixel")
plt.show()

## Further reading

- `notebooks/scripts/pubfigs/` - the publication figure suite built on these tools
- `.documents/DETECTOR_MODELS.md` - all eight detector models
- `notebooks/docs/OPTIMIZATION_RESULTS.md` - the full calibration campaign notes